# 🧠 Naive Bayes & NLP — Concepts Notebook
### Sample Data se Theory Samjho

---

**Topics Covered:**
1. 📐 Bayes Theorem — Probability ka Foundation
2. 🔢 Naive Bayes Algorithm — Theory aur Manual Calculation
3. 📝 Introduction to NLP — Text Data kaise handle karein
4. 🛍️ Bag of Words aur TF-IDF
5. 💬 Text Classification with Naive Bayes
6. 🎯 Gaussian NB vs Multinomial NB vs Bernoulli NB
7. 🏆 Capstone Exercise

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
print('✅ Ready!')

---
## 📐 Section 1: Bayes Theorem

**Naye Keywords:**
- **Probability** — kisi event ke hone ki chance (0 se 1)
- **Prior Probability P(C)** — kisi bhi information ke bina class ka probability
- **Likelihood P(X|C)** — agar class C hai toh feature X milne ka probability
- **Posterior P(C|X)** — feature X dekhne ke baad class C ka updated probability
- **Bayes Theorem** — prior + evidence se posterior calculate karna

In [ ]:
# --- Bayes Theorem: Simple Example ---
# Scenario: Email spam detection
# P(Spam) = 0.3  (30% emails spam hote hain)
# P(Ham)  = 0.7  (70% emails normal hote hain)
# P('FREE'|Spam) = 0.8  (spam mein 'FREE' word 80% baar aata hai)
# P('FREE'|Ham)  = 0.1  (normal email mein 'FREE' word 10% baar aata hai)

P_spam = 0.3
P_ham = 0.7
P_free_given_spam = 0.8
P_free_given_ham = 0.1

# P('FREE') = P('FREE'|Spam)*P(Spam) + P('FREE'|Ham)*P(Ham)
P_free = P_free_given_spam * P_spam + P_free_given_ham * P_ham

# Bayes Theorem: P(Spam|'FREE') = P('FREE'|Spam) * P(Spam) / P('FREE')
P_spam_given_free = (P_free_given_spam * P_spam) / P_free
P_ham_given_free  = (P_free_given_ham  * P_ham)  / P_free

print("=" * 45)
print("Bayes Theorem Example: Email Spam")
print("=" * 45)
print(f"P('FREE') = {P_free:.3f}")
print(f"P(Spam | email has 'FREE') = {P_spam_given_free:.3f}  → {P_spam_given_free*100:.1f}%")
print(f"P(Ham  | email has 'FREE') = {P_ham_given_free:.3f}  → {P_ham_given_free*100:.1f}%")
print()
print("Decision: Email is", 'SPAM' if P_spam_given_free > P_ham_given_free else 'HAM')

### 🎯 Bayes Theorem Formula

$$P(C|X) = \frac{P(X|C) \cdot P(C)}{P(X)}$$

- **P(C|X)** = Posterior — feature X dekhke C ka probability
- **P(X|C)** = Likelihood — class C mein feature X ka probability  
- **P(C)** = Prior — class C ka starting probability
- **P(X)** = Evidence — normalization factor (constant for all classes)

**Real Life:** Doctor pehle se jaanta hai ki 10% log diabetic hain (prior). Blood test result dekhke posterior update hota hai.

---
## 🔢 Section 2: Naive Bayes Algorithm — Manual Calculation

**Naya Keyword:**
- **Naive Assumption** — sab features ek doosre se independent hain (practically galat, par kaam karta hai!)
- **Multinomial NB** — count data ke liye (word counts, text)
- **Gaussian NB** — continuous features ke liye (normal distribution assume karta hai)
- **Bernoulli NB** — binary features ke liye (word present/absent)

In [ ]:
# --- Sample Dataset: Fruit Classification ---
fruit_data = pd.DataFrame({
    'Color': ['Red','Red','Yellow','Yellow','Orange','Red','Yellow','Orange'],
    'Size':  ['Big','Big','Small','Big','Big','Small','Small','Small'],
    'Fruit': ['Apple','Apple','Banana','Banana','Orange','Apple','Banana','Orange']
})

print("Fruit Dataset:")
print(fruit_data)
print("\nFruit counts:", fruit_data['Fruit'].value_counts().to_dict())

In [ ]:
# --- Manual Naive Bayes Step-by-Step ---
# Question: Color=Red, Size=Small — kaunsa fruit?

total = len(fruit_data)
fruits = fruit_data['Fruit'].value_counts()

print("Step 1: Prior Probabilities")
priors = {f: fruits[f]/total for f in fruits.index}
for f, p in priors.items():
    print(f"  P({f}) = {fruits[f]}/{total} = {p:.3f}")

print("\nStep 2: Likelihoods for Color=Red, Size=Small")
query_color = 'Red'
query_size  = 'Small'

scores = {}
for fruit in fruits.index:
    subset = fruit_data[fruit_data['Fruit'] == fruit]
    n = len(subset)
    p_color = len(subset[subset['Color'] == query_color]) / n
    p_size  = len(subset[subset['Size']  == query_size])  / n
    p_prior = priors[fruit]
    score   = p_color * p_size * p_prior
    scores[fruit] = score
    print(f"  P(Red|{fruit})={p_color:.2f}, P(Small|{fruit})={p_size:.2f}, Prior={p_prior:.2f} → Score={score:.4f}")

winner = max(scores, key=scores.get)
print(f"\n✅ Prediction: {winner} (highest score = {scores[winner]:.4f})")

### 🎯 Naive Bayes Formula (Multi-feature)

$$P(C|x_1, x_2, ..., x_n) \propto P(C) \cdot \prod_{i=1}^{n} P(x_i|C)$$

**Naive** kyun kehte hain? Kyunki hum assume karte hain:
$$P(x_1, x_2 | C) = P(x_1|C) \cdot P(x_2|C)$$
Yeh practically galat hota hai (features correlated hote hain), par algorithm surprisingly well kaam karta hai!

---
## 📝 Section 3: Introduction to NLP

**NLP = Natural Language Processing** — computers ko human language samjhana

**Naye Keywords:**
- **Corpus** — text documents ka collection
- **Token** — ek word ya character
- **Tokenization** — sentence ko words mein todna
- **Stop Words** — common words jo meaning nahi dete (the, is, a, an)
- **Stemming** — word ko root form mein laana (running → run)
- **Lemmatization** — word ka proper base form (better → good)

In [ ]:
# --- Basic NLP Operations (without NLTK) ---
import re

sample_text = "The quick brown fox jumps over the lazy dog! Python is amazing."

# Tokenization
tokens = sample_text.lower().split()
print("Tokens:", tokens[:8], '...')

# Remove punctuation
tokens_clean = [re.sub(r'[^\w]', '', t) for t in tokens if re.sub(r'[^\w]', '', t)]
print("Clean tokens:", tokens_clean)

# Stop word removal (simple list)
stop_words = {'the', 'is', 'a', 'an', 'over', 'of', 'in', 'and', 'or'}
filtered = [t for t in tokens_clean if t not in stop_words]
print("After removing stop words:", filtered)

# Word frequency
from collections import Counter
freq = Counter(tokens_clean)
print("\nWord frequencies:")
print(dict(sorted(freq.items(), key=lambda x: -x[1])[:5]))

---
## 🛍️ Section 4: Bag of Words aur TF-IDF

**Naye Keywords:**
- **Bag of Words (BoW)** — har document ko word counts ka vector banana
- **Vocabulary** — sab unique words ka set
- **CountVectorizer** — BoW sklearn mein
- **TF-IDF** — Term Frequency-Inverse Document Frequency
  - **TF** — ek document mein word kitni baar aaya
  - **IDF** — word kitne rare hai across all documents
  - Rare useful words ko zyada weight milta hai
- **Sparse Matrix** — mostly zeros wali matrix (zyada features, kam values)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Sample documents
docs = [
    "I love playing cricket",
    "Cricket is a great sport",
    "I love coding in Python",
    "Python is a great language"
]

# --- Bag of Words ---
cv = CountVectorizer()
bow_matrix = cv.fit_transform(docs)

bow_df = pd.DataFrame(bow_matrix.toarray(), columns=cv.get_feature_names_out())
bow_df.index = [f'Doc{i+1}' for i in range(len(docs))]
print("Bag of Words Matrix:")
print(bow_df)

print("\nVocabulary size:", len(cv.vocabulary_))

In [ ]:
# --- TF-IDF ---
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(docs)

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray().round(3),
    columns=tfidf.get_feature_names_out()
)
tfidf_df.index = [f'Doc{i+1}' for i in range(len(docs))]
print("TF-IDF Matrix:")
print(tfidf_df)
print()
print("Note: 'cricket' has high TF-IDF in Doc1&2 (specific to those docs)")
print("Note: 'is' has lower TF-IDF (appears in many docs = less informative)")

**TF-IDF Formula:**

$$\text{TF-IDF}(t, d) = \text{TF}(t,d) \times \text{IDF}(t)$$

$$\text{IDF}(t) = \log\left(\frac{N}{1 + df_t}\right) + 1$$

Jahan N = total documents, df_t = documents jisme term t hai

---
## 💬 Section 5: Text Classification with Naive Bayes

**Naye Keywords:**
- **MultinomialNB** — word counts ke saath kaam karta hai (text ke liye best)
- **GaussianNB** — continuous numerical features ke liye
- **BernoulliNB** — binary data (word present/absent)
- **Pipeline** — preprocessing + model ek saath chain karna
- **Laplace Smoothing (alpha)** — zero probability avoid karna

In [ ]:
from sklearn.naive_bayes import MultinomialNB, GaussianNB, BernoulliNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

# --- Sample SMS Dataset ---
messages = [
    ("Win a FREE iPhone now! Click here", "spam"),
    ("Get FREE money! You won a prize!", "spam"),
    ("Cheap loans! Call now for FREE cash", "spam"),
    ("WINNER: You have been selected for a cash prize", "spam"),
    ("Exclusive deal! Limited time offer FREE gift", "spam"),
    ("Hey, are you coming to dinner tonight?", "ham"),
    ("Can you pick up milk from the store?", "ham"),
    ("Meeting postponed to next Tuesday", "ham"),
    ("Happy birthday! Hope you have a great day", "ham"),
    ("I will call you after the match", "ham"),
    ("Reminder: your appointment is tomorrow at 3pm", "ham"),
    ("Congratulations! FREE entry in our prize draw", "spam"),
    ("Are you free this weekend for a walk?", "ham"),
    ("Buy now, pay later! FREE delivery", "spam"),
    ("How was your exam today?", "ham")
]

texts, labels = zip(*messages)
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.3, random_state=42
)

print("Train samples:", len(X_train))
print("Test samples:", len(X_test))

In [ ]:
# --- Pipeline: TF-IDF + MultinomialNB ---
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('nb', MultinomialNB(alpha=1.0))   # alpha = Laplace smoothing
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print("Predictions:")
for text, actual, pred in zip(X_test, y_test, y_pred):
    status = '✅' if actual == pred else '❌'
    print(f"{status} Actual={actual:4s} Pred={pred:4s} | {text[:45]}")

print("\nAccuracy:", pipeline.score(X_test, y_test))

In [ ]:
# --- Classification Report ---
print(classification_report(y_test, y_pred))

# Confusion Matrix
ConfusionMatrixDisplay.from_estimator(
    pipeline, X_test, y_test,
    display_labels=['ham', 'spam'], cmap='Blues'
)
plt.title('Naive Bayes — SMS Spam Detection')
plt.tight_layout()
plt.show()

In [ ]:
# --- Predict new messages ---
new_messages = [
    "You have won FREE tickets to the concert!",
    "See you at lunch tomorrow?",
    "Claim your FREE prize now!"
]

predictions = pipeline.predict(new_messages)
probabilities = pipeline.predict_proba(new_messages)

print("New Message Predictions:")
for msg, pred, prob in zip(new_messages, predictions, probabilities):
    print(f"  {pred.upper():4s} (ham={prob[0]:.2f}, spam={prob[1]:.2f}) | {msg[:50]}")

---
## 🎯 Section 6: GaussianNB — Numerical Data

**GaussianNB** continuous numerical features ke liye use hota hai.
Assumption: features normally distributed hain given each class.

$$P(x_i|C) = \frac{1}{\sqrt{2\pi\sigma_C^2}} \exp\left(-\frac{(x_i - \mu_C)^2}{2\sigma_C^2}\right)$$

In [ ]:
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler

# Iris dataset
iris = load_iris()
X_iris, y_iris = iris.data, iris.target
X_tr, X_te, y_tr, y_te = train_test_split(X_iris, y_iris, test_size=0.2, random_state=42)

gnb = GaussianNB()
gnb.fit(X_tr, y_tr)
gnb_acc = gnb.score(X_te, y_te)

print("GaussianNB on Iris:")
print(f"Accuracy: {gnb_acc:.4f}")
print("\nClass means learned by GaussianNB:")
means_df = pd.DataFrame(gnb.theta_, columns=iris.feature_names, index=iris.target_names)
print(means_df.round(2))

---
## 🏆 Capstone Exercise

**Movie Review Sentiment Analysis**
Niche diya gaya data hai movie reviews ka. Aapko positive/negative classify karna hai.

In [ ]:
movie_reviews = [
    ("Absolutely amazing film! Great acting and story", "positive"),
    ("Terrible movie, waste of time and money", "negative"),
    ("Loved every minute, highly recommend!", "positive"),
    ("Boring and predictable, very disappointing", "negative"),
    ("Brilliant screenplay and fantastic direction", "positive"),
    ("Awful acting, terrible plot, do not watch", "negative"),
    ("One of the best movies I have seen this year", "positive"),
    ("Completely unwatchable, worst movie ever", "negative"),
    ("Stunning visuals, great performance", "positive"),
    ("Dull and lifeless, disappointed everyone", "negative"),
    ("Masterpiece! Outstanding in every way", "positive"),
    ("Hated it, nothing interesting happens", "negative")
]

rev_texts, rev_labels = zip(*movie_reviews)

# TODO A: Train-test split karo
# X_tr_m, X_te_m, y_tr_m, y_te_m = ...

# TODO B: TF-IDF + MultinomialNB pipeline banao
# ...

# TODO C: Accuracy aur classification report print karo
# ...

# TODO D: 3 naye reviews predict karo
# new_reviews = ["Incredible movie!", "Total disaster", "Worth watching"]
# ...

print("Capstone ready — TODO A se D complete karo!")

---
## ✅ Capstone Solutions

Pehle khud try karo. Solutions niche hain:

In [ ]:
# SOLUTION A: Train-test split
X_tr_m, X_te_m, y_tr_m, y_te_m = train_test_split(
    rev_texts, rev_labels, test_size=0.3, random_state=42
)

# SOLUTION B: Pipeline
movie_pipe = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('nb', MultinomialNB(alpha=0.5))
])
movie_pipe.fit(X_tr_m, y_tr_m)

# SOLUTION C
print("Accuracy:", movie_pipe.score(X_te_m, y_te_m))
print(classification_report(y_te_m, movie_pipe.predict(X_te_m)))

# SOLUTION D
new_reviews = ["Incredible movie!", "Total disaster", "Worth watching"]
preds = movie_pipe.predict(new_reviews)
for r, p in zip(new_reviews, preds):
    print(f"  {p.upper():8s} | {r}")

---
## 🎓 Summary — Aaj Humne Kya Seekha?

| Concept | Matlab | Use Case |
|---|---|---|
| Bayes Theorem | Prior + Evidence → Posterior | Medical diagnosis, spam |
| Naive Assumption | Features independent hain | Simplifies computation |
| Bag of Words | Text → word count vector | Text classification |
| TF-IDF | Rare useful words ko zyada weight | Search engines, NLP |
| MultinomialNB | Text classification | Spam, sentiment |
| GaussianNB | Continuous data | Medical, sensor data |
| Pipeline | Preprocessing + Model chain | Production ML |
| Laplace Smoothing | Zero probability avoid karna | Unseen words handle |

### 🌍 Real Life Applications:
- **Gmail spam filter** — Naive Bayes use karta hai
- **News categorization** — Sports, Politics, Tech automatically classify
- **Medical diagnosis** — symptoms se disease probability
- **Sentiment analysis** — product reviews positive/negative